# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Dataset ID (@id): {getattr(metadata, '@id', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, their `@id`s, and all columns for exploration.

We use `dataset.record_sets` to enumerate record sets and their structure.

**All referencing uses `@id` fields.**

In [ ]:
# List available record sets, their fields and columns
record_sets = list(dataset.record_sets)
print(f"Total record sets found: {len(record_sets)}\n")

for idx, record_set in enumerate(record_sets):
    print(f"[{idx}] RecordSet: 'name' = {record_set.name}\n   @id = {record_set['@id']}")
    print(f"   Description: {getattr(record_set, 'description', '')}")
    # List fields
    if hasattr(record_set, 'fields'):
        print("   Fields:")
        for field in record_set.fields:
            print(f"     - name: {getattr(field, 'name', '')}")
            print(f"         @id: {field['@id']}")
            print(f"         dataType: {getattr(field, 'dataType', '')}")
    # List columns
    if hasattr(record_set, 'columns'):
        print("   Columns:")
        for col in record_set.columns:
            print(f"     - {getattr(col, 'name', '')} (@id: {col['@id']})")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the `@id`s from the overview above to reference record sets and fields.

We will load each available record set and preview their columns.

In [ ]:
# Collect record set @ids for extraction
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    # Extract records from this record set
    records = list(dataset.records(record_set=record_set_id))
    # Sanity check - only include if not empty
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"RecordSet '@id': {record_set_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head())
        print("\n---\n")

## 4. Exploratory Data Analysis (EDA)
### Filtering, normalization, and grouping

Let's select a numeric field from a record set and perform some EDA operations. **We reference fields by their `@id`**.

For this example, pick a RecordSet with numeric fields such as abundance, MW, or coverage (adjust field `@id` as needed).

In [ ]:
# Please set these to the proper @id from the overview section, e.g.,
# example_record_set_id = 'https://api.app.sen.science/frontiers/...record-set-id...'
# example_numeric_field_id = 'https://api.app.sen.science/frontiers/...field-id...'

# For illustration, set to the first non-empty record set and a numeric column
if dataframes:
    example_record_set_id = list(dataframes.keys())[0]
    df = dataframes[example_record_set_id]
    # Find possible numeric fields by dtype
    numeric_field_candidates = df.select_dtypes(include=['number']).columns.tolist()
    if numeric_field_candidates:
        numeric_field = numeric_field_candidates[0]
        print(f"Using record set: {example_record_set_id}\nNumeric field: {numeric_field}")
        threshold = df[numeric_field].mean()  # e.g., mean as threshold
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by another categorical field, e.g., 'Sample' or similar
        possible_group_fields = df.select_dtypes(include=['object']).columns.tolist()
        group_field = None
        for field in possible_group_fields:
            if field.lower().startswith('sample') or field.lower().startswith('group'):
                group_field = field
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"\nGrouped data by {group_field} (mean of {numeric_field}):")
            print(grouped_df.head())
    else:
        print("No numeric fields found in selected record set for EDA.")
else:
    print("No record sets with data available for EDA.")

## 5. Visualization
Visualize distributions or relationships between numeric fields.

Here, we plot a histogram and a boxplot for the selected numeric field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field' in locals():
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field}")

    plt.subplot(1,2,2)
    sns.boxplot(x=df[numeric_field].dropna())
    plt.title(f"Boxplot of {numeric_field}")
    plt.show()
else:
    print("No data/numeric field available for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to access and explore the Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells dataset using the `mlcroissant` library.

We loaded the dataset, enumerated all record sets and field `@id`s, and demonstrated basic exploratory data analysis and visualizations on the numeric data, referencing all entities by `@id`.

For further domain-specific analysis, select relevant fields and grouping columns as indicated in the schema overview, always referencing by their unique `@id`.